# C-03 Data Quality and Representativeness Audit

**Control:** C-03 Data Quality & Representativeness Audit, MAK-AI Responsible AI Toolkit.

**Learning goal:** run a structured audit of a dataset (types, missing values, duplicates, invalid values, outliers) and compare who is in the data with the population the model will serve.

**Time to run:** about 1 minute on an ordinary laptop. No GPU.

**Licence:** this notebook is Apache-2.0.

**Example data:**
- Diabetes 130-US Hospitals for Years 1999-2008 (Strack and others, 2014), DOI 10.24432/C5230J, licence CC BY 4.0, 101,766 hospital stays, from https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008.
- Reference population: World Bank, World Development Indicators, population estimates and projections, Uganda 2023, by age band and sex (api.worldbank.org v2, retrieved 30 Oct 2026, data last updated 1 Jul 2026), licence CC BY 4.0. A small extract is stored inside the notebook, with its date, so that it runs offline.

Both are credited here because CC BY 4.0 asks for attribution.

**A caveat before you start:** the stays are from United States hospitals, so a Ugandan population is the wrong thing to compare them with if you want a fair picture of that hospital. It is used here on purpose, to show what a mismatch between data and reference looks like, and to make the point of the "Stop and decide" cell: the reference has to be the people your model will serve.

## Linked risk areas and questions

Generated from content 5.0.0. Do not edit by hand.

Risk areas: 6

- PP-02: Data Scarcity & Availability Bias
- PP-03: Historical & Societal Bias
- PP-04: Population Bias
- PP-05: Sampling & Generalization Bias
- PP-07: Labeling Bias
- IP-07: Data Integrity Compromise

Questions: 16: Q-PP-03, Q-PP-04, Q-PP-05, Q-PP-24, Q-PP-09, Q-PP-11, Q-PP-12, Q-PP-16, Q-PP-17, Q-PP-18, Q-PP-19, Q-PP-20, Q-PP-22, Q-PP-23, Q-IP-11, Q-IP-22

## When to use this notebook

Use it when you are answering questions about the quality and coverage of your data and your status is **Not yet** or **Partly**, especially with the reason code `ny_needs_skills` ("Needs skills or guidance the team does not have yet"). It helps with:

- `Q-PP-16`: have you analysed the data (options such as class balance and distribution across subgroups)?
- `Q-PP-09` and `Q-PP-22`: do you know how complete and how representative the data is?
- `Q-IP-11`: have you checked the data you train on against what you described?

You need a table with one row per record, and some idea of the population the system will serve.

In [ ]:
import hashlib
import importlib.metadata
import io
import platform
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chisquare

SEED = 20261030  # the one random seed; this notebook has no random step, but the seed is kept for the record
np.random.seed(SEED)
pd.set_option('display.width', 150)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 80)

PACKAGES = ['pandas', 'numpy', 'scipy', 'matplotlib']

print('Python', platform.python_version())
for _package in PACKAGES:
    print(_package, importlib.metadata.version(_package))

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/296/data.csv'
DATA_SHA256 = 'f792c388d9b470aac4fbf21c39b176534d55a116e7626881c30bbe7ab0341422'
DATA_FILE = 'uci-diabetes-130-296-data.csv'  # the cache file name is fixed here

CACHE_DIR = Path.home() / '.cache' / 'makrai-notebooks'


def fetch_checked(url, sha256, file_name):
    """Return the bytes of a dataset whose SHA-256 matches `sha256`.

    The file is downloaded once into the cache folder, and written there only after its hash
    matches. Every run, including the first, hashes the bytes before they are used, and a
    mismatch stops the notebook. `file_name` is fixed in this notebook, never taken from the address.
    """
    if not url.startswith('https://'):
        raise ValueError(f'the dataset address must be https: {url}')
    path = CACHE_DIR / file_name
    if path.exists():
        data, source, remedy = path.read_bytes(), f'the cached file {path}', f'Delete {path} and run again.'
    else:
        request = urllib.request.Request(url, headers={'User-Agent': 'makrai-notebooks'})
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                if not response.geturl().startswith('https://'):
                    raise RuntimeError(f'{url} redirected to a non-https address: {response.geturl()}')
                data = response.read()
        except OSError as error:
            raise RuntimeError(f'could not download {url}: {error}. If you cannot reach it, save the file by hand '
                               f'as {path}; it is checked against the pinned SHA-256 either way.') from error
        source, remedy = url, 'Do not use this download; check the dataset page for a new version.'
    found = hashlib.sha256(data).hexdigest()
    if found != sha256:
        raise RuntimeError(f'{file_name}: the SHA-256 of {source} is {found}, expected {sha256}. {remedy}')
    if not path.exists():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + '.part')
        partial.write_bytes(data)
        partial.replace(path)  # the cache never holds a half-written file
    return data


# To use your own file, replace the next line with:  raw = Path('your-file.csv').read_bytes()
raw = fetch_checked(DATA_URL, DATA_SHA256, DATA_FILE)
DATA_SHA256_USED = hashlib.sha256(raw).hexdigest()  # the evidence summary records the hash of the bytes used
df = pd.read_csv(io.BytesIO(raw), low_memory=False)
print(f'{len(df):,} rows, {df.shape[1]} columns; SHA-256 of the data used: {DATA_SHA256_USED[:12]}...')

# The reference population: persons in Uganda in 2023 by five-year age band and sex, World Bank WDI (CC BY 4.0).
# Retrieved 30 Oct 2026 from api.worldbank.org/v2, indicators SP.POP.<band>.<FE or MA>, data last updated 1 Jul 2026.
WDI_UGANDA_2023 = {
    '0004.FE': 3928044,
    '0509.FE': 3488212,
    '1014.FE': 3193173,
    '1519.FE': 2849671,
    '2024.FE': 2465959,
    '2529.FE': 2071087,
    '3034.FE': 1643786,
    '3539.FE': 1227332,
    '4044.FE': 911790,
    '4549.FE': 730504,
    '5054.FE': 599730,
    '5559.FE': 460326,
    '6064.FE': 339739,
    '6569.FE': 245913,
    '7074.FE': 170391,
    '7579.FE': 93029,
    '80UP.FE': 115305,
    '0004.MA': 4010306,
    '0509.MA': 3543476,
    '1014.MA': 3215557,
    '1519.MA': 2840777,
    '2024.MA': 2443087,
    '2529.MA': 2042460,
    '3034.MA': 1608486,
    '3539.MA': 1192230,
    '4044.MA': 877334,
    '4549.MA': 693423,
    '5054.MA': 554417,
    '5559.MA': 403671,
    '6064.MA': 275624,
    '6569.MA': 185426,
    '7074.MA': 120961,
    '7579.MA': 62333,
    '80UP.MA': 53042,
}
BANDS = [('[0-10)', ['0004', '0509']), ('[10-20)', ['1014', '1519']), ('[20-30)', ['2024', '2529']),
         ('[30-40)', ['3034', '3539']), ('[40-50)', ['4044', '4549']), ('[50-60)', ['5054', '5559']),
         ('[60-70)', ['6064', '6569']), ('[70-80)', ['7074', '7579']), ('[80-100)', ['80UP'])]  # the source stops at 80 and over
reference_table = pd.DataFrame(
    [{'age': label, 'gender': sex_label, 'population': sum(WDI_UGANDA_2023[f'{b}.{code}'] for b in parts)}
     for label, parts in BANDS for sex_label, code in [('Female', 'FE'), ('Male', 'MA')]])
print(f'reference: {reference_table.population.sum():,} people in {len(reference_table)} groups')

In [ ]:
# ADAPT HERE. Edit only this cell to run the notebook on your own data. Every later cell reads only these names.
ROW_ID_COLUMN = 'encounter_id'       # the column that is different on every row (one stay, one record)
PERSON_ID_COLUMN = 'patient_nbr'     # the column that identifies the same person across rows
GROUP_COLUMNS = ['race', 'gender', 'age']   # columns to compare missing values across
RANGE_RULES = {'time_in_hospital': (1, 14), 'num_lab_procedures': (1, 100),
               'num_medications': (1, 60), 'number_diagnoses': (1, 16)}   # allowed range of each numeric column
ALLOWED_CODES = {'gender': ['Female', 'Male'], 'readmitted': ['NO', '>30', '<30']}   # allowed values of each coded column
OUTLIER_COLUMNS = ['time_in_hospital', 'num_lab_procedures', 'num_medications', 'number_inpatient']
IQR_MULTIPLIER = 1.5                 # a value this many interquartile ranges beyond the quartiles is flagged
MISSING_GAP = 0.05                   # missing shares that differ between groups by this much are flagged
MIN_GROUP_SIZE = 100                 # groups smaller than this are flagged, not interpreted
MAKEUP_COLUMNS = ['age', 'gender']   # the columns that define the groups compared with the reference
ALIGN_VALUES = {'[80-90)': '[80-100)', '[90-100)': '[80-100)'}   # merges data categories so they match the reference
REFERENCE_TABLE = reference_table    # a table with the MAKEUP_COLUMNS and one column of people counts
REFERENCE_COUNT_COLUMN = 'population'   # the column of REFERENCE_TABLE that holds the counts

### Step 1: types and schema

**What this does.** It lists every column with its type, how many different values it has, how much of it is missing and an example. It also marks columns that look like codes stored as numbers (a small set of whole numbers in a column whose name ends in `_id`), because a code such as an admission type should never be averaged.

**What it shows.** Whether the columns are what you think they are: numbers stored as text, codes stored as numbers, a column that is constant, an identifier that is unique for every row.

**What it does not show.** Whether the values are right. A column can be perfectly typed and wrong.

In [ ]:
schema = pd.DataFrame({
    'type': df.dtypes.astype(str),
    'distinct values': df.nunique(),
    'missing share': df.isna().mean().round(3),
    'example': df.iloc[0].astype(str),
})
looks_like_code = df.columns.str.endswith('_id') & (df.nunique().to_numpy() <= 30) & df.dtypes.astype(str).str.startswith('int').to_numpy()
schema['note'] = np.where(looks_like_code, 'a code stored as a number: do not average it', '')
schema.loc[df.nunique() == 1, 'note'] = 'constant: carries no information'
print(f'{len(df):,} rows, {df.shape[1]} columns')
schema

### Step 2: missing values, by column and by group

**What this does.** It reports the share missing in each column, then asks whether missingness is **systematic**: for each column with some missing values, it compares the missing share across the groups in `GROUP_COLUMNS` (leaving out groups smaller than `MIN_GROUP_SIZE`) and flags the column when the largest gap between two groups is at least `MISSING_GAP`.

**What it shows.** Which columns are mostly empty, and where the holes are not spread evenly. A column that is missing for 90 percent of one age band and 30 percent of another was not filled in by chance; dropping the rows with holes would quietly remove one kind of person.

**What it does not show.** Why the values are missing. A value can be missing because it was not collected, not applicable, refused, or lost, and each calls for a different decision.

In [ ]:
missing_by_column = df.isna().mean().sort_values(ascending=False)
with_holes = missing_by_column[missing_by_column >= 0.01].index.tolist()

gap_rows = []
for group_column in GROUP_COLUMNS:
    labels = df[group_column].fillna('missing').astype(str)
    big = labels.map(labels.value_counts()) >= MIN_GROUP_SIZE
    shares = df.loc[big, with_holes].isna().groupby(labels[big]).mean()
    for column in with_holes:
        if column == group_column:
            continue
        gap_rows.append({'column': column, 'compared across': group_column, 'groups compared': len(shares),
                         'lowest missing share': shares[column].min(), 'highest missing share': shares[column].max(),
                         'gap': shares[column].max() - shares[column].min()})
missing_by_group = pd.DataFrame(gap_rows).sort_values('gap', ascending=False).reset_index(drop=True)
missing_by_group['systematic?'] = np.where(missing_by_group['gap'] >= MISSING_GAP, 'yes, look at it', '')

print('share missing, columns with at least 1 percent missing:')
print(missing_by_column[with_holes].round(3).to_string())
print()
print('is missingness the same in every group? (largest gaps first)')
print(missing_by_group.head(12).round(3).to_string(index=False))

### Step 3: duplicates, and the difference between a duplicate and a repeat visit

**What this does.** Three counts.

1. **Exact duplicates**: rows identical in every column.
2. **Duplicates apart from the row identifier**: rows identical in everything except the column that is unique to each row. These are copies that got a new number.
3. **Repeat visits**: rows that belong to a person who already appears in an earlier row.

**What it shows.** A duplicate is a data fault: remove or explain it. A repeat visit is a real event, and it is not a fault, but it matters: when the same person appears many times, the rows are not independent, counts of "people" are really counts of visits, and a train/test split that scatters one person's visits across both parts lets the model be tested on people it has already seen.

**What it does not show.** Near duplicates that differ by a typo (a misspelt name, a changed date of birth) or rows for the same person under two identifiers. Finding those takes matching rules that depend on your data.

In [ ]:
exact_duplicates = int(df.duplicated().sum())
copies_with_new_id = int(df.drop(columns=[ROW_ID_COLUMN]).duplicated().sum())
same_details_different_people = int(df.drop(columns=[ROW_ID_COLUMN, PERSON_ID_COLUMN]).duplicated().sum())
repeat_visit_rows = int(df[PERSON_ID_COLUMN].duplicated().sum())
visits_per_person = df[PERSON_ID_COLUMN].value_counts()

duplicates = pd.Series({
    'exact duplicate rows': exact_duplicates,
    'rows identical apart from the row identifier': copies_with_new_id,
    'rows identical apart from both identifiers (same details, could be different people)': same_details_different_people,
    'rows that are a repeat visit of someone already seen': repeat_visit_rows,
    'people seen more than once': int((visits_per_person > 1).sum()),
    'most rows for one person': int(visits_per_person.max()),
})
duplicates.to_frame('count')

### Step 4: validity rules

**What this does.** It checks each row against rules you write in the "Adapt here" cell: a **range** for numeric columns and a list of **allowed values** for coded columns. It counts the rows that break each rule. A value that is missing is not counted here, because Step 2 already counted it.

**What it shows.** Impossible or unexpected values: a negative age, an unrecorded sex, a code that is not on the list.

**What it does not show.** That every value inside the range is right. The rules here are illustrative, loosely based on the dataset's description. Real rules should come from the people who collect and use the data, and should be written down before you look at the counts.

In [ ]:
rule_rows = []
for column, (low, high) in RANGE_RULES.items():
    broken = df[column].notna() & ~df[column].between(low, high)
    rule_rows.append({'rule': f'{column} between {low} and {high}', 'rows that break it': int(broken.sum()), 'share of rows': broken.mean()})
for column, allowed in ALLOWED_CODES.items():
    broken = df[column].notna() & ~df[column].isin(allowed)
    rule_rows.append({'rule': f'{column} is one of {allowed}', 'rows that break it': int(broken.sum()), 'share of rows': broken.mean()})
validity = pd.DataFrame(rule_rows)
print(validity.round(4).to_string(index=False))

### Step 5: outliers are flagged, never deleted

**What this does.** For each column in `OUTLIER_COLUMNS` it finds the quartiles, and flags any value further than `IQR_MULTIPLIER` times the interquartile range beyond them. Flagged rows stay in the data; the cell checks that no row was removed.

**What it shows.** Where unusual values sit, so that someone can look at them.

**What it does not show.** That a flagged value is wrong. A very long hospital stay is rare and real. For counts where most values are zero, the rule flags every non-zero value, which is a sign the rule does not suit the column. Deleting outliers without looking changes who is in the data, and the people with the most extreme circumstances are often the ones a system most needs to get right.

In [ ]:
rows_before = len(df)
flag_rows = []
outlier_flags = pd.DataFrame(index=df.index)
for column in OUTLIER_COLUMNS:
    q1, q3 = df[column].quantile([0.25, 0.75])
    spread = q3 - q1
    low, high = q1 - IQR_MULTIPLIER * spread, q3 + IQR_MULTIPLIER * spread
    outlier_flags[column] = (df[column] < low) | (df[column] > high)
    flag_rows.append({'column': column, 'lower fence': low, 'upper fence': high,
                      'rows flagged': int(outlier_flags[column].sum()), 'share flagged': outlier_flags[column].mean()})
outliers = pd.DataFrame(flag_rows)
assert len(df) == rows_before  # nothing is removed
print(f'rows still in the data: {len(df):,} (nothing was removed)')
print(outliers.round(3).to_string(index=False))

### Step 6: who is in the data, compared with who the model will serve

**What this does.** It lines the data's groups (here age band by gender) up with a reference population: the same groups, with the number of people in each. It then compares the two on **shares**:

- the share of the data in each group, and the share of the reference;
- the **difference** in percentage points, and the ratio;
- a **chi-square goodness-of-fit test** (SciPy's `chisquare`), which asks whether the data's group counts could have been drawn from the reference shares. With a hundred thousand rows the test rejects almost any difference, so its p-value is not the useful part;
- the **total variation distance**: half the sum of the absolute differences in share. It is the share of people who would have to move to a different group to make the data match the reference, and it does not grow with the number of rows.

Groups that are too small in the data, or where fewer than 5 people would be expected, are flagged and not interpreted. Rows whose group does not exist in the reference are counted separately.

**What it shows.** Which groups are over- or under-represented compared with the reference.

**What it does not show.** That the reference is the right one. Compare a hospital's patients with a national census and every difference looks like a failure of the data, when the hospital may simply serve different people. That choice is the next cell.

In [ ]:
aligned = df[MAKEUP_COLUMNS].copy()
for column in MAKEUP_COLUMNS:
    aligned[column] = aligned[column].replace(ALIGN_VALUES)

reference = REFERENCE_TABLE.set_index(MAKEUP_COLUMNS)[REFERENCE_COUNT_COLUMN].astype(float)
in_data = aligned.groupby(MAKEUP_COLUMNS).size()
not_in_reference = in_data[~in_data.index.isin(reference.index)]
compared = in_data.reindex(reference.index).fillna(0)

makeup = pd.DataFrame({'people in data': compared, 'people in reference': reference})
makeup['data share'] = makeup['people in data'] / makeup['people in data'].sum()
makeup['reference share'] = makeup['people in reference'] / makeup['people in reference'].sum()
makeup['difference (points)'] = 100 * (makeup['data share'] - makeup['reference share'])
makeup['data share / reference share'] = makeup['data share'] / makeup['reference share']
expected = makeup['reference share'] * makeup['people in data'].sum()
makeup['flag'] = np.where(makeup['people in data'] < MIN_GROUP_SIZE, 'small in the data',
                          np.where(expected < 5, 'under 5 expected', ''))

statistic, p_value = chisquare(f_obs=makeup['people in data'], f_exp=expected)
total_variation = 0.5 * float((makeup['data share'] - makeup['reference share']).abs().sum())
print(f'rows compared: {int(compared.sum()):,}; rows in groups the reference does not have: {int(not_in_reference.sum())} {({' / '.join(map(str, k)): int(v) for k, v in not_in_reference.items()})}')
print(f'chi-square goodness of fit: statistic {statistic:,.0f}, degrees of freedom {len(makeup) - 1}, p-value {p_value:.3g}')
print(f'total variation distance: {total_variation:.3f} (about {total_variation:.0%} of people would have to move group to match the reference)')
print()
print(makeup.round(3).to_string())

fig, axis = plt.subplots(figsize=(7, 6))
positions = np.arange(len(makeup))
axis.barh(positions - 0.2, makeup['data share'], height=0.4, label='data')
axis.barh(positions + 0.2, makeup['reference share'], height=0.4, label='reference')
axis.set_yticks(positions)
axis.set_yticklabels([' / '.join(map(str, i)) for i in makeup.index], fontsize=8)
axis.invert_yaxis()
axis.set_xlabel('share of people')
axis.set_title('Who is in the data, and who is in the reference')
axis.legend()
fig.tight_layout()

### Step 7 (optional): build the reference for your own country

**This cell is optional and is skipped in the automatic runs, because it needs the internet and the World Bank's figures can be revised.** It fetches the population by five-year age band and sex for a country from the World Bank's public API, using the same indicators as the stored extract. Name your country's three-letter code and the year in the first two lines, run the cell, then use the table it builds as `REFERENCE_TABLE` in the "Adapt here" cell, after merging age bands so that they match your data. Always say in your record where the reference came from and on what date.

In [ ]:
import json

COUNTRY_CODE = 'UGA'   # three-letter code of the country the model will serve
REFERENCE_YEAR = 2023
fetched = []
for band in ['0004', '0509', '1014', '1519', '2024', '2529', '3034', '3539', '4044', '4549', '5054', '5559',
             '6064', '6569', '7074', '7579', '80UP']:
    for sex_label, code in zip(sorted(REFERENCE_TABLE[MAKEUP_COLUMNS[1]].unique()), ['FE', 'MA']):   # the source codes: FE female, MA male
        url = f'https://api.worldbank.org/v2/country/{COUNTRY_CODE}/indicator/SP.POP.{band}.{code}?date={REFERENCE_YEAR}&format=json'
        with urllib.request.urlopen(url, timeout=60) as response:
            meta, rows = json.load(response)
        fetched.append({'band': band, MAKEUP_COLUMNS[1]: sex_label, REFERENCE_COUNT_COLUMN: rows[0]['value']})
print('World Bank data last updated', meta['lastupdated'])
pd.DataFrame(fetched).head()

### Stop and decide: which reference is right, and what to do about what you found

The notebook compared the data with the reference you gave it. It cannot tell you whether that reference is the right one.

Questions to ask in the room:

1. Who will the model serve? A hospital's patients are not the national census, and the national census is not the same as the people who reach the clinic. The reference should be the people the model will be used on. If you do not have figures for them, write down what you used instead and why.
2. If the data does not look like the reference, is that a problem or a fact about who comes through the door? What would change if the data matched?
3. Which columns have too many holes to use? For the ones with systematic gaps (Step 2), who is being lost, and is that acceptable?
4. What will you do about the invalid values and the duplicates: fix at the source, remove, or leave and note? Who decides?
5. Which small groups did the table leave uninterpreted, and how will you make sure someone still looks at them?

Write your choices in the next cell. Leave it blank if you have not decided, and the summary will say so.

In [ ]:
# YOUR CHOICES. The notebook does not fill this in. Plain words are fine.
REFERENCE_WE_CHOSE = ''         # the population you compared with, its source and date
WHY_THIS_REFERENCE = ''         # why it is the people the model will serve
GAPS_IN_THE_DATA_WE_ACCEPT = ''  # missing or unrepresented groups you accept, and why
ISSUES_WE_WILL_FIX = ''         # duplicates, invalid values or columns you will fix or drop, and who decides

In [ ]:
EVIDENCE_DIR = Path('evidence')
EVIDENCE_DIR.mkdir(exist_ok=True)

# Small tables of counts and shares. They hold no row of the data.
schema[['type', 'distinct values', 'missing share', 'note']].to_csv(EVIDENCE_DIR / 'C-03_schema.csv')
missing_by_group.round(4).to_csv(EVIDENCE_DIR / 'C-03_missing_by_group.csv', index=False)
duplicates.to_frame('count').to_csv(EVIDENCE_DIR / 'C-03_duplicates.csv')
validity.round(4).to_csv(EVIDENCE_DIR / 'C-03_validity.csv', index=False)
outliers.round(4).to_csv(EVIDENCE_DIR / 'C-03_outliers.csv', index=False)
makeup.round(4).to_csv(EVIDENCE_DIR / 'C-03_makeup.csv')
fig.savefig(EVIDENCE_DIR / 'C-03_makeup.png', dpi=100, bbox_inches='tight')

versions = '\n'.join(f'- {package} {importlib.metadata.version(package)}' for package in PACKAGES)
systematic = missing_by_group[missing_by_group['gap'] >= MISSING_GAP]
summary = f"""# C-03 Data Quality and Representativeness Audit: evidence summary

Attach this file, with the CSV and PNG files beside it, to the decision it supports (questions Q-PP-09, Q-PP-16, Q-PP-22, Q-IP-11). It holds counts and shares only, never the data.

## What was audited
- Rows: {len(df):,}; columns: {df.shape[1]}
- Duplicates: {exact_duplicates} exact; {copies_with_new_id} apart from the row identifier; {repeat_visit_rows:,} rows are repeat visits of people already seen (not faults, but the rows are not independent)
- Rows breaking a validity rule: {int(validity['rows that break it'].sum()):,} in total across {len(validity)} rules
- Outliers flagged (nothing removed): {int(outliers['rows flagged'].sum()):,} flags across {len(outliers)} columns
- Missing: {len(with_holes)} columns have at least 1 percent missing; {len(systematic)} column-and-group comparisons differ by {MISSING_GAP:.0%} or more

## Make-up against the reference
- Groups compared: {', '.join(MAKEUP_COLUMNS)}; {len(makeup)} groups; groups with fewer than {MIN_GROUP_SIZE} people are flagged and not interpreted
- Total variation distance: {total_variation:.3f}
- Chi-square goodness of fit: statistic {statistic:,.0f}, p-value {p_value:.3g} (with this many rows the p-value is not informative)

## Choices made in the stop-and-decide cell
- Reference chosen: {REFERENCE_WE_CHOSE or 'not written yet'}
- Why: {WHY_THIS_REFERENCE or 'not written yet'}
- Gaps we accept: {GAPS_IN_THE_DATA_WE_ACCEPT or 'not written yet'}
- Issues we will fix: {ISSUES_WE_WILL_FIX or 'not written yet'}

## Files beside this summary
C-03_schema.csv, C-03_missing_by_group.csv, C-03_duplicates.csv, C-03_validity.csv, C-03_outliers.csv, C-03_makeup.csv, C-03_makeup.png

## Package versions
- Python {platform.python_version()}
{versions}

## Data
SHA-256 of the data file used: {DATA_SHA256_USED}
Reference: World Bank WDI, Uganda 2023, retrieved 30 Oct 2026, data last updated 1 Jul 2026, CC BY 4.0 (stored in the notebook)
"""
(EVIDENCE_DIR / 'C-03_summary.md').write_text(summary, encoding='utf-8')
print(summary)

## What this does not show, and where to go next

- **It does not say the data is good enough.** It says what is there. Whether the data supports the use is a judgment, informed by the people who collected it.
- **It cannot see what was never collected.** People who never reached the hospital, groups not recorded, and columns that were never asked for are invisible to every table here.
- **Counts are not causes.** Systematic missingness says the gaps are not random. It does not say why.
- **The reference is a choice.** A different reference gives a different picture. The record should show which one was used, from where and on what date.
- **Labels and measures can be wrong in ways no rule catches.** A rule can confirm that an age is between 0 and 120. It cannot confirm that it is this person's age.

Where to go next: C-04 Bias Detection in Data (label rates, proxies and annotator agreement), C-05 Data Distribution Analysis (groups, intersections and class balance), and the datasheet template, which records what you learned about the data's origin, gaps and uses.